In [2]:
# CELL 1 — Setup, GPU check, load IMDB data
!pip install transformers -q

import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from transformers import DistilBertTokenizerFast, DistilBertForSequenceClassification
from sklearn.model_selection import train_test_split
import time, pickle, json

print("CUDA available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device name:", torch.cuda.get_device_name(0))
device = "cuda" if torch.cuda.is_available() else "cpu"
print("using device:", device)

INTERIM_PATH = "/kaggle/input/datasets/anurajgogoi/cineiq-interim-files/interim"

CUDA available: True
device name: Tesla T4
using device: cuda


In [3]:
imdb = pd.read_parquet(f"{INTERIM_PATH}/imdb_reviews_clean.parquet")
imdb["label"] = (imdb["sentiment"] == "positive").astype(int)
print("imdb shape:", imdb.shape)

train_df, val_df = train_test_split(imdb, test_size=0.15, random_state=42, stratify=imdb["label"])
train_df, val_df = train_df.reset_index(drop=True), val_df.reset_index(drop=True)
print("train:", train_df.shape, "val:", val_df.shape)

imdb shape: (49582, 3)
train: (42144, 3) val: (7438, 3)


In [4]:
# CELL 2 — Dataset class + full-scale DataLoaders

tokenizer = DistilBertTokenizerFast.from_pretrained("distilbert-base-uncased")

class IMDBDataset(Dataset):
    def __init__(self, texts, labels, tokenizer, max_length=256):
        self.texts = texts.reset_index(drop=True)
        self.labels = labels.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        encoding = self.tokenizer(
            self.texts.iloc[idx], truncation=True, padding="max_length",
            max_length=self.max_length, return_tensors="pt"
        )
        return {
            "input_ids": encoding["input_ids"].squeeze(0),
            "attention_mask": encoding["attention_mask"].squeeze(0),
            "label": torch.tensor(self.labels.iloc[idx], dtype=torch.long),
        }

train_dataset = IMDBDataset(train_df["review_clean"], train_df["label"], tokenizer, max_length=256)
val_dataset = IMDBDataset(val_df["review_clean"], val_df["label"], tokenizer, max_length=256)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=16)

print("train batches:", len(train_loader))
print("val batches:", len(val_loader))

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

train batches: 2634
val batches: 465


In [4]:
# CELL 3 — Load DistilBERT, train on full data

def train_distilbert(model, train_loader, val_loader, num_epochs=3, lr=2e-5, device="cpu"):
    model = model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr)
    history = {"train_loss": [], "val_loss": [], "val_accuracy": []}

    for epoch in range(num_epochs):
        model.train()
        total_train_loss, n_batches = 0.0, 0
        epoch_start = time.time()

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            n_batches += 1

        train_loss = total_train_loss / n_batches

        model.eval()
        total_val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_val_loss += outputs.loss.item()
                preds = outputs.logits.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_loss = total_val_loss / len(val_loader)
        val_acc = correct / total
        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(f"epoch {epoch+1}/{num_epochs} — train loss: {train_loss:.4f} — "
              f"val loss: {val_loss:.4f} — val acc: {val_acc:.4f} — {time.time()-epoch_start:.1f}s")

    return model, history


model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

start = time.time()
model, history = train_distilbert(model, train_loader, val_loader, num_epochs=3, lr=2e-5, device=device)
print(f"\ntotal training time: {time.time()-start:.1f}s")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.bias   | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
pre_classifier.weight   | MISSING    | 
classifier.bias         | MISSING    | 
pre_classifier.bias     | MISSING    | 
classifier.weight       | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/3 — train loss: 0.2539 — val loss: 0.2193 — val acc: 0.9123 — 1101.2s
epoch 2/3 — train loss: 0.1505 — val loss: 0.2083 — val acc: 0.9204 — 1107.1s
epoch 3/3 — train loss: 0.0798 — val loss: 0.2827 — val acc: 0.9181 — 1107.2s

total training time: 3315.8s


In [5]:
# CELL — train_distilbert with checkpointing (keeps the best-val-loss epoch, not necessarily the final one)

def train_distilbert(model, train_loader, val_loader, num_epochs=4, lr=2e-5, patience=2, device="cpu"):
    """
    Fine-tunes DistilBERT with early stopping on validation loss.
    Saves the state dict after every epoch that improves val loss, and
    restores that checkpoint at the end — protects against the
    overfitting pattern observed in the first run (val loss started
    climbing again at epoch 3 while train loss kept falling).
    """
    model = model.to(device)
    optimizer = AdamW(model.parameters(), lr=lr)

    history = {"train_loss": [], "val_loss": [], "val_accuracy": []}
    best_val_loss = float("inf")
    epochs_without_improvement = 0
    best_state = None
    best_epoch = None

    for epoch in range(num_epochs):
        model.train()
        total_train_loss, n_batches = 0.0, 0
        epoch_start = time.time()

        for batch in train_loader:
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels = batch["label"].to(device)

            optimizer.zero_grad()
            outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
            loss = outputs.loss
            loss.backward()
            optimizer.step()

            total_train_loss += loss.item()
            n_batches += 1

        train_loss = total_train_loss / n_batches

        model.eval()
        total_val_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for batch in val_loader:
                input_ids = batch["input_ids"].to(device)
                attention_mask = batch["attention_mask"].to(device)
                labels = batch["label"].to(device)
                outputs = model(input_ids=input_ids, attention_mask=attention_mask, labels=labels)
                total_val_loss += outputs.loss.item()
                preds = outputs.logits.argmax(dim=1)
                correct += (preds == labels).sum().item()
                total += labels.size(0)

        val_loss = total_val_loss / len(val_loader)
        val_acc = correct / total

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_accuracy"].append(val_acc)

        print(f"epoch {epoch+1}/{num_epochs} — train loss: {train_loss:.4f} — "
              f"val loss: {val_loss:.4f} — val acc: {val_acc:.4f} — {time.time()-epoch_start:.1f}s")

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_epoch = epoch + 1
            epochs_without_improvement = 0
            # Move to CPU when checkpointing to avoid doubling GPU memory usage
            best_state = {k: v.clone().cpu() for k, v in model.state_dict().items()}
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"\nearly stopping at epoch {epoch+1} (no improvement for {patience} epochs)")
                break

    if best_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_state.items()})
        print(f"\nrestored best checkpoint — epoch {best_epoch}, val loss: {best_val_loss:.4f}")

    return model, history, best_epoch


In [6]:
# CELL — Train with checkpointing (up to 4 epochs, patience=2)

model = DistilBertForSequenceClassification.from_pretrained("distilbert-base-uncased", num_labels=2)

start = time.time()
model, history, best_epoch = train_distilbert(
    model, train_loader, val_loader, num_epochs=4, lr=2e-5, patience=2, device=device
)
print(f"\ntotal training time: {time.time()-start:.1f}s")
print(f"best epoch: {best_epoch}")

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/268M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

DistilBertForSequenceClassification LOAD REPORT from: distilbert-base-uncased
Key                     | Status     | 
------------------------+------------+-
vocab_layer_norm.weight | UNEXPECTED | 
vocab_projector.bias    | UNEXPECTED | 
vocab_transform.bias    | UNEXPECTED | 
vocab_transform.weight  | UNEXPECTED | 
vocab_layer_norm.bias   | UNEXPECTED | 
pre_classifier.bias     | MISSING    | 
pre_classifier.weight   | MISSING    | 
classifier.weight       | MISSING    | 
classifier.bias         | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


epoch 1/4 — train loss: 0.2537 — val loss: 0.2138 — val acc: 0.9165 — 1004.9s
epoch 2/4 — train loss: 0.1520 — val loss: 0.2000 — val acc: 0.9247 — 1008.1s
epoch 3/4 — train loss: 0.0822 — val loss: 0.2408 — val acc: 0.9230 — 1008.0s
epoch 4/4 — train loss: 0.0468 — val loss: 0.3151 — val acc: 0.9193 — 1007.4s

early stopping at epoch 4 (no improvement for 2 epochs)

restored best checkpoint — epoch 2, val loss: 0.2000

total training time: 4029.3s
best epoch: 2


In [7]:
# CELL — Save the classifier, tokenizer, and results

model.save_pretrained("/kaggle/working/distilbert_sentiment")
tokenizer.save_pretrained("/kaggle/working/distilbert_sentiment")

sentiment_results = {
    "model": "DistilBERT fine-tuned (distilbert-base-uncased)",
    "training_data": "IMDB 50K Reviews (Week 1 cleaned, 49,582 after dedup)",
    "train_size": len(train_df),
    "val_size": len(val_df),
    "max_seq_length": 256,
    "learning_rate": 2e-5,
    "best_epoch": best_epoch,
    "val_accuracy": round(history["val_accuracy"][best_epoch - 1], 4),
    "val_loss": round(history["val_loss"][best_epoch - 1], 4),
    "epoch_history": history,
    "note": "Trained with early stopping (patience=2) on validation loss. "
            "Overfitting appeared reliably at epoch 3 across two separate "
            "training runs -- val loss climbed while train loss kept "
            "falling. Best checkpoint (epoch 2) is what's saved here, "
            "consistent with the same generalization-over-final-epoch "
            "principle applied to the GRU in Week 2.",
}

with open("/kaggle/working/sentiment_results.json", "w") as f:
    json.dump(sentiment_results, f, indent=2)

print(json.dumps({k: v for k, v in sentiment_results.items() if k != "epoch_history"}, indent=2))
print("\nsaved: distilbert_sentiment/ (model + tokenizer), sentiment_results.json")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "model": "DistilBERT fine-tuned (distilbert-base-uncased)",
  "training_data": "IMDB 50K Reviews (Week 1 cleaned, 49,582 after dedup)",
  "train_size": 42144,
  "val_size": 7438,
  "max_seq_length": 256,
  "learning_rate": 2e-05,
  "best_epoch": 2,
  "val_accuracy": 0.9247,
  "val_loss": 0.2,
  "note": "Trained with early stopping (patience=2) on validation loss. Overfitting appeared reliably at epoch 3 across two separate training runs -- val loss climbed while train loss kept falling. Best checkpoint (epoch 2) is what's saved here, consistent with the same generalization-over-final-epoch principle applied to the GRU in Week 2."
}

saved: distilbert_sentiment/ (model + tokenizer), sentiment_results.json
